In [4]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer , BitsAndBytesConfig
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",          # NormalFloat 4-bit quantization
#     bnb_4bit_use_double_quant=True,     # Nested quantization
#     bnb_4bit_compute_dtype=torch.bfloat16 # Speeds up matrix multiplication
# )


# ---------- Load & split document ----------
document = PyPDFLoader(r"C:\Users\MY PC\Pictures\documents\Autonomous Revenue Platform Master PRDBRD.pdf")
document = document.load()
print(len(document))
print(document[0].page_content)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=180
)
chunks = splitter.split_documents(document)

# ---------- Embeddings + vector store ----------
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vec_database = FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

retriever = vec_database.as_retriever(
    search_kwargs={"k": 5},   # fixed typo: was "search_kwards"
    search_type="similarity"
)

# ---------- Model (CPU-only, no bitsandbytes) ----------
model_name = "google/gemma-3-1b-it"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "5GiB", "cpu": "14GiB"},   # set the GPU number to a bit under your VRAM
    low_cpu_mem_usage=True,
)



tokenizer = AutoTokenizer.from_pretrained(model_name)

# # model = AutoModelForCausalLM.from_pretrained(
# #     model_name,
# #     low_cpu_mem_usage=True
# # )


36
AUTONOMOUS REVENUE &
CUSTOMER LIFECYCLE PLATFORM
MASTER PRODUCT REQUIREMENTS DOCUMENT
& BUSINESS REQUIREMENTS DOCUMENT
Final Build-Ready Master Version
Version 3.0 — September 2026
Document Classification: Technical Product Specification & Implementation Baseline
Target Audience: System Architects, Automation Engineers, AI Engineers, Product
Teams
Orchestration Standard: n8n Micro-Workflow Engine
AI Architecture: Local-First Multi-LLM with Elastic Cloud Fallback and RAG Verification
Scope Boundary: Full Lifecycle (Lead Discovery to Conversion, Retention, and Renewal)
Autonomous Technical Product Specification • Zero Brand Dependencies


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [15]:
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# ---------- Prompt ----------
SYSTEM_PROMPT = """

you are a helpfull assistance. 

"""

def build_chat_prompt(context: str, question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ---------- Generation (streaming) ----------
def generate_answer(inputs: dict) -> str:
    context = inputs["context"]
    question = inputs["question"]

    prompt_text = build_chat_prompt(context, question)

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer   # prints tokens live to stdout as they're generated
    )

    generated_tokens = output_ids[0][model_inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

llm = RunnableLambda(generate_answer)

from operator import itemgetter

# ---------- RAG chain ----------
rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question")
    }
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke({"question": "what is Business Outcomes"})
print("\n\nFinal answer:", response)


Business Outcomes:

Here’s a breakdown of the Business Outcomes you’ve listed, categorized for clarity:

**Core Business Goals:**

*   **Continuous Influx:** Generating qualified prospects continuously across diverse channels.
*   **Upsell & Cross-sell Engine:** Identifying and recommending higher-tier service plans.
*   **Referral Engine:** Identifying and nurturing satisfied clients through referrals.
*   **Renewal & Retention:** Managing contracts and ensuring client satisfaction.

**Operational & System-Related Outcomes:**

*   **CRM Integration & Orchestration:**  A centralized CRM and orchestration engine to manage the entire process.
*   **Workflow Integrity Monitoring:** Ensuring smooth and efficient operational workflows.
*   **Data Deduplication & Cleansing:** Removing duplicate records and improving data quality.
*   **Delivery Audits:** Monitoring and auditing the delivery process.

**Specific Agent-Focused Outcomes:**

*   **Meeting/MOM Agent Transcription & Analysis:**  A

In [16]:
from IPython.display import display , Markdown

display(Markdown(rag_chain.invoke({"question" :"FinalProductDefnition"})))

Okay, let’s break down the provided information and answer your question.

**Understanding the Document & the Core Message**

The document outlines a framework for building and managing a sophisticated Revenue & Customer Lifecycle Platform (R&CPL) – essentially a system for managing customer interactions and driving revenue. It emphasizes a proactive, automated approach to engagement.  It’s structured around key themes:

* **Focus on Automation:**  The system is designed to automate key processes like lead capture, proposal generation, and follow-up.
* **Proactive Engagement:**  It’s built to anticipate customer needs and proactively engage with them.
* **Risk Management:**  It incorporates safeguards against negative interactions (dormancy, threats) and robust monitoring.
* **Data-Driven Optimization:**  It tracks and analyzes interactions to continuously improve performance.

**Analyzing Your Question: "FinalProductDefnition"**

You've asked: "FinalProductDefnition" – this is the *ti

Okay, let’s break down the provided information and answer your question.

**Understanding the Document & the Core Message**

The document outlines a framework for building and managing a sophisticated Revenue & Customer Lifecycle Platform (R&CPL) – essentially a system for managing customer interactions and driving revenue. It emphasizes a proactive, automated approach to engagement.  It’s structured around key themes:

* **Focus on Automation:**  The system is designed to automate key processes like lead capture, proposal generation, and follow-up.
* **Proactive Engagement:**  It’s built to anticipate customer needs and proactively engage with them.
* **Risk Management:**  It incorporates safeguards against negative interactions (dormancy, threats) and robust monitoring.
* **Data-Driven Optimization:**  It tracks and analyzes interactions to continuously improve performance.

**Analyzing Your Question: "FinalProductDefnition"**

You've asked: "FinalProductDefnition" – this is the *title* of a section within the document.  Let's analyze what it's likely meant to be:

**The "Final Product Definition" section is a foundational document outlining the *requirements* for a specific product – a robust and automated platform for managing customer relationships and revenue.** It's essentially a *blueprint* for the system's core functionality.

**Here's a breakdown of the key elements and what they signify:**

* **Core Functionality (

In [ ]:
#"google/gemma-3-12b-it"

non duplicates {1122}
